In [1]:
import csv
import random
import datetime as dt
from pathlib import Path

In [2]:
# PARAMETRY LABORATORIUM
N_ROWS = 100_000  # na start; potem łatwo zwiększyć
OUTPUT_DIR = Path.cwd().parent.parent.parent / "common" / "data" / "events"
OUTPUT_FILE = OUTPUT_DIR / "events.csv"

random.seed(42)

In [3]:
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_FILE

PosixPath('/home/aligator/projects/agh/dbids/common/data/events/events.csv')

In [4]:
countries = ["PL", "DE", "FR", "IT", "ES", "US", "GB", "SE", "NO", "NL"]
devices = ["mobile", "desktop", "tablet"]
event_types = ["view", "add_to_cart", "purchase"]

start_time = dt.datetime(2025, 1, 1)

In [5]:
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    # nagłówek – WSPÓLNY dla ClickHouse i Postgres
    writer.writerow(
        [
            "event_time",
            "user_id",
            "session_id",
            "product_id",
            "price",
            "quantity",
            "country",
            "device",
            "event_type",
        ]
    )

    for _ in range(N_ROWS):
        event_time = start_time + dt.timedelta(
            seconds=random.randint(0, 180 * 24 * 3600)
        )

        user_id = random.randint(1, 50_000)
        session_id = random.randint(1, 200_000)
        product_id = random.randint(1, 10_000)
        price = round(random.uniform(5, 500), 2)
        quantity = 1 if random.random() < 0.85 else random.randint(2, 5)
        country = random.choice(countries)
        device = random.choice(devices)

        r = random.random()
        event_type = "purchase" if r < 0.05 else "add_to_cart" if r < 0.20 else "view"

        writer.writerow(
            [
                event_time.strftime("%Y-%m-%d %H:%M:%S"),
                user_id,
                session_id,
                product_id,
                price,
                quantity,
                country,
                device,
                event_type,
            ]
        )


In [6]:
# pokaż pierwsze 5 linii
with open(OUTPUT_FILE, encoding="utf-8") as f:
    for i in range(5):
        print(f.readline().strip())


event_time,user_id,session_id,product_id,price,quantity,country,device,event_type
2025-05-05 03:56:41,7297,6557,4507,126.22,1,DE,tablet,view
2025-04-16 21:35:32,5698,154795,6913,20.73,1,IT,tablet,view
2025-04-19 23:35:29,13032,187701,8929,212.66,1,ES,mobile,view
2025-02-01 00:03:58,45754,110786,5575,142.55,1,US,mobile,add_to_cart


In [7]:
import pandas as pd

df = pd.read_csv(OUTPUT_FILE)
df.head()

,event_time,user_id,session_id,product_id,price,quantity,country,device,event_type
0,2025-05-05 03:56:41,7297,6557,4507,126.22,1,DE,tablet,view
1,2025-04-16 21:35:32,5698,154795,6913,20.73,1,IT,tablet,view
2,2025-04-19 23:35:29,13032,187701,8929,212.66,1,ES,mobile,view
3,2025-02-01 00:03:58,45754,110786,5575,142.55,1,US,mobile,add_to_cart
4,2025-01-19 18:43:51,23527,90166,9892,135.94,1,SE,tablet,add_to_cart


Pokaż rozkład częstości kategorii event_type

In [8]:
df["event_type"].value_counts(normalize=True)

event_type
view           0.79712
add_to_cart    0.15196
purchase       0.05092
Name: proportion, dtype: float64

In [9]:
len(df)

100000